# LeetCode 1222 — Queens That Can Attack the King bằng Python

Link đề: https://leetcode.com/problems/queens-that-can-attack-the-king/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Simulation

Skeleton LeetCode:

```python
class Solution:
    def queensAttacktheKing(self, queens: List[List[int]], king: List[int]) -> List[List[int]]:
        ...
```

Ghi chú: kết quả là **danh sách toạ độ** các quân hậu tấn công trực tiếp được vua; **thứ tự tuỳ ý** (khi kiểm tra ta sắp xếp trước khi so sánh).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Mô hình hoá nước đi của quân **hậu** (queen): theo hàng, cột và hai đường chéo — tổng cộng **8 hướng** biểu diễn bằng vector `(dx, dy)` với `dx, dy ∈ {−1, 0, 1}`, không cùng bằng 0.
2. Hiểu khái niệm **"bị chắn"**: chỉ quân hậu **gần vua nhất** trên mỗi hướng mới tấn công được.
3. Đi từ **kiểm tra từng quân hậu + kiểm tra vật cản** (`O(q²)`) → **mô phỏng từ vua đi ra 8 hướng** (`O(8·8)`) → **gom nhóm theo hướng, giữ quân gần nhất** (`O(q)`).
4. Dùng `set` các tuple để tra cứu "ô này có hậu không" trong `O(1)`.
5. Tránh các lỗi: quên `break` (lấy cả hậu bị chắn), quên đường chéo phụ, chuẩn hoá hướng sai, so sánh kết quả mà không sắp xếp.

## 2. Đề bài & ví dụ

Trên bàn cờ `8 × 8` (đánh chỉ số từ 0) có nhiều **quân hậu đen** và **một quân vua trắng**. Cho mảng `queens` với `queens[i] = [xQueen_i, yQueen_i]` là vị trí quân hậu thứ `i`, và mảng `king = [xKing, yKing]` là vị trí vua.

Trả về toạ độ các quân hậu đen **tấn công trực tiếp** được vua. Thứ tự tuỳ ý.

Hậu tấn công trực tiếp vua khi nằm **cùng hàng, cùng cột hoặc cùng đường chéo** với vua và **không có quân nào khác chắn** giữa hai quân.

**Ví dụ 1:** `queens = [[0,1],[1,0],[4,0],[0,4],[3,3],[2,4]], king = [0,0]` → `[[0,1],[1,0],[3,3]]`.

**Ví dụ 2:** `queens = [[0,0],[1,1],[2,2],[3,4],[3,5],[4,4],[4,5]], king = [3,3]` → `[[2,2],[3,4],[4,4]]`.

**Ràng buộc:**
- `1 <= queens.length < 64`
- `queens[i].length == king.length == 2`
- `0 <= xQueen_i, yQueen_i, xKing, yKing < 8`
- Mọi vị trí đều **khác nhau**.

In [1]:
import random
import time
from typing import List

examples = [
    ([[0, 1], [1, 0], [4, 0], [0, 4], [3, 3], [2, 4]], [0, 0], [[0, 1], [1, 0], [3, 3]]),
    ([[0, 0], [1, 1], [2, 2], [3, 4], [3, 5], [4, 4], [4, 5]], [3, 3], [[2, 2], [3, 4], [4, 4]]),
]


def show_board(queens, king, attackers=()):
    qs, at = {tuple(q) for q in queens}, {tuple(a) for a in attackers}
    for x in range(8):
        row = []
        for y in range(8):
            if [x, y] == list(king):
                row.append("K")
            elif (x, y) in at:
                row.append("Q")      # hậu tấn công được
            elif (x, y) in qs:
                row.append("q")      # hậu không tấn công được
            else:
                row.append(".")
        print(" ".join(row))


for qs, k, want in examples:
    show_board(qs, k, want)
    print("->", want, "\n")

K Q . . q . . .
Q . . . . . . .
. . . . q . . .
. . . Q . . . .
q . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
-> [[0, 1], [1, 0], [3, 3]] 

q . . . . . . .
. q . . . . . .
. . Q . . . . .
. . . K Q q . .
. . . . Q q . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
-> [[2, 2], [3, 4], [4, 4]] 



Trong bảng trên: `K` là vua, `Q` là hậu **tấn công được**, `q` là hậu **không** tấn công được (không cùng đường, hoặc bị chắn).

## 3. Kiến thức nền

### 3.1 Toạ độ và 8 hướng

Ô `(x, y)`: `x` là hàng (từ trên xuống), `y` là cột (từ trái sang). Hậu đi theo 8 vector hướng:

| Hướng | (dx, dy) |
|---|---|
| lên / xuống | (−1, 0) / (1, 0) |
| trái / phải | (0, −1) / (0, 1) |
| chéo chính | (−1, −1) / (1, 1) |
| chéo phụ | (−1, 1) / (1, −1) |

Sinh gọn bằng hai vòng lặp `dx, dy ∈ {−1, 0, 1}` và bỏ `(0, 0)`.

In [2]:
DIRS8 = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if (dx, dy) != (0, 0)]
print(len(DIRS8), DIRS8)

8 [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


### 3.2 Khi nào hai ô cùng hàng / cột / chéo?

Với hai ô `(x1, y1)`, `(x2, y2)` khác nhau, đặt `dx = x2 − x1`, `dy = y2 − y1`:
- Cùng hàng: `dx == 0`.
- Cùng cột: `dy == 0`.
- Cùng đường chéo (chính **hoặc** phụ): `|dx| == |dy|`.

In [3]:
def same_line(a, b):
    dx, dy = b[0] - a[0], b[1] - a[1]
    return dx == 0 or dy == 0 or abs(dx) == abs(dy)


k = (3, 3)
for q in [(3, 7), (0, 3), (5, 5), (5, 1), (4, 6)]:
    print(q, same_line(k, q))

(3, 7) True
(0, 3) True
(5, 5) True
(5, 1) True
(4, 6) False


### 3.3 Chuẩn hoá hướng bằng hàm dấu `sign`

Nếu hai ô cùng một đường, hướng **đơn vị** từ `a` đến `b` là `(sign(dx), sign(dy))` với `sign(t) ∈ {−1, 0, 1}`. Khoảng cách (số bước) là `max(|dx|, |dy|)`. Python không có `sign` sẵn — viết `(t > 0) - (t < 0)`.

In [4]:
def sign(t):
    return (t > 0) - (t < 0)


for q in [(3, 7), (0, 3), (5, 1)]:
    dx, dy = q[0] - 3, q[1] - 3
    print(q, "hướng", (sign(dx), sign(dy)), "khoảng cách", max(abs(dx), abs(dy)))

(3, 7) hướng (0, 1) khoảng cách 4
(0, 3) hướng (-1, 0) khoảng cách 3
(5, 1) hướng (1, -1) khoảng cách 2


### 3.4 `set` các tuple

List không băm được (không làm khoá `set`/`dict`), nên ta đổi `[x, y]` thành tuple `(x, y)`. Khi trả kết quả cho LeetCode thì đổi lại thành list `[x, y]`.

In [5]:
queens = [[0, 1], [1, 0]]
qset = {tuple(q) for q in queens}
print(qset, (0, 1) in qset, (2, 2) in qset)
try:
    {[0, 1]}
except TypeError as e:
    print("list làm phần tử set:", e)

{(0, 1), (1, 0)} True False
list làm phần tử set: unhashable type: 'list'


## 4. Ý tưởng nền tảng / trực giác

Đặt mình vào vị trí **vua** và nhìn ra 8 hướng như tia sáng. Trên mỗi hướng, **quân hậu đầu tiên** tia gặp sẽ chặn tia — quân đó tấn công được vua, mọi quân phía sau bị nó che.

Ví dụ 1, vua ở `(0, 0)` (góc trên trái) nên chỉ có 3 hướng đi vào bàn cờ:
- Sang phải `(0, 1)`: gặp `(0,1)` ngay → tấn công. `(0,4)` phía sau bị che.
- Xuống `(1, 0)`: gặp `(1,0)` ngay → tấn công. `(4,0)` bị che.
- Chéo `(1, 1)`: `(1,1)` trống, `(2,2)` trống, `(3,3)` có hậu → tấn công.
- `(2,4)` không nằm trên đường nào của vua.

Kết quả `[[0,1],[1,0],[3,3]]` — khớp đề.

In [6]:
qs, k, _ = examples[0]
for q in qs:
    print(q, "cùng đường với vua?", same_line(k, q))

[0, 1] cùng đường với vua? True
[1, 0] cùng đường với vua? True
[4, 0] cùng đường với vua? True
[0, 4] cùng đường với vua? True
[3, 3] cùng đường với vua? True
[2, 4] cùng đường với vua? False


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi quân hậu `q`:
1. Nếu `q` không cùng đường với vua → bỏ.
2. Ngược lại, kiểm tra **mọi** quân hậu khác `p` xem có nằm **giữa** `q` và vua không. Nếu không có → `q` tấn công được.

`p` nằm giữa `q` và vua khi `p` cùng hướng từ vua với `q` và gần vua hơn.

In [7]:
def attack_brute(queens, king):
    kx, ky = king

    def direction_and_dist(q):
        dx, dy = q[0] - kx, q[1] - ky
        if not (dx == 0 or dy == 0 or abs(dx) == abs(dy)):
            return None
        return (sign(dx), sign(dy)), max(abs(dx), abs(dy))

    ans = []
    for q in queens:
        dq = direction_and_dist(q)
        if dq is None:
            continue
        blocked = False
        for p in queens:
            if p is q:
                continue
            dp = direction_and_dist(p)
            if dp is not None and dp[0] == dq[0] and dp[1] < dq[1]:
                blocked = True
                break
        if not blocked:
            ans.append(q)
    return ans


for qs, k, want in examples:
    got = attack_brute(qs, k)
    print(sorted(got))
    assert sorted(got) == sorted(want)

[[0, 1], [1, 0], [3, 3]]
[[2, 2], [3, 4], [4, 4]]


**Độ phức tạp:** Time `O(q²)` với `q` là số hậu (`< 64` → tối đa ~4000 phép kiểm tra, vẫn rất nhanh). Space `O(1)` ngoài kết quả.

**Nút thắt:** với bàn cờ lớn hơn (ví dụ `n × n`, `q` tới `10^5`) thì `O(q²)` không ổn. Và ta lặp lại việc "so với mọi quân khác" dù chỉ cần biết quân **gần nhất** trên mỗi hướng.

## 6. Bước đột phá

**Insight:** có **tối đa 8** quân hậu tấn công được vua — mỗi hướng tối đa một quân, là quân **gần nhất**. Vậy thay vì hỏi "quân này có bị chắn không?", hãy hỏi "trên hướng này, quân gần nhất là ai?".

Hai cách hiện thực:

1. **Mô phỏng từ vua:** với mỗi hướng `(dx, dy)`, bước từng ô `(kx + t·dx, ky + t·dy)`, `t = 1, 2, ...`.
   - **Nếu** ra ngoài bàn cờ **thì** hướng này không có ai, chuyển hướng.
   - **Nếu** ô có hậu **thì** ghi nhận và **dừng** hướng này (`break`) — mọi quân sau bị chắn.
2. **Gom nhóm theo hướng:** với mỗi quân hậu cùng đường với vua, tính `(hướng, khoảng cách)`; giữ quân có khoảng cách **nhỏ nhất** cho từng hướng.

**Bất biến của mô phỏng:** khi đang ở bước `t` trên hướng `d`, mọi ô ở bước `1..t−1` trên hướng đó đều **trống** — nên nếu ô bước `t` có hậu thì nó không bị chắn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng từ vua đi ra 8 hướng (lời giải chuẩn)

In [8]:
def attack_simulate(queens, king):
    qset = {tuple(q) for q in queens}
    kx, ky = king
    ans = []
    for dx, dy in DIRS8:
        x, y = kx + dx, ky + dy
        while 0 <= x < 8 and 0 <= y < 8:
            if (x, y) in qset:
                ans.append([x, y])
                break                    # quân đầu tiên chắn mọi quân phía sau
            x += dx
            y += dy
    return ans


for qs, k, want in examples:
    got = attack_simulate(qs, k)
    print(got)
    assert sorted(got) == sorted(want)

[[0, 1], [1, 0], [3, 3]]
[[2, 2], [3, 4], [4, 4]]


**Độ phức tạp:** Time `O(q + 8·N)` với `N = 8` là cạnh bàn cờ (dựng `set` mất `O(q)`, mỗi hướng tối đa `N−1` bước). Space `O(q)` cho `set`.

### 7.2 Cách 3 — Mô phỏng trên ma trận boolean 8×8

Thay `set` bằng ma trận `board[x][y]` (True nếu có hậu). Với bàn cờ nhỏ cố định, đây là cách "trực quan" nhất và không cần băm.

In [9]:
def attack_board(queens, king):
    board = [[False] * 8 for _ in range(8)]
    for x, y in queens:
        board[x][y] = True
    ans = []
    for dx, dy in DIRS8:
        x, y = king[0] + dx, king[1] + dy
        while 0 <= x < 8 and 0 <= y < 8 and not board[x][y]:
            x += dx
            y += dy
        if 0 <= x < 8 and 0 <= y < 8:          # dừng vì gặp hậu, không phải vì ra biên
            ans.append([x, y])
    return ans


for qs, k, want in examples:
    assert sorted(attack_board(qs, k)) == sorted(want)
print("Cách 3 đúng trên ví dụ")

Cách 3 đúng trên ví dụ


**Độ phức tạp:** Time `O(N² + q + 8N)` (khởi tạo bàn cờ `N²`), Space `O(N²)`.

### 7.3 Cách 4 — Gom theo hướng, giữ quân gần nhất (không phụ thuộc kích thước bàn cờ)

Duyệt mỗi quân hậu **một lần**: nếu cùng đường với vua, tính `(hướng, khoảng cách)`; dùng `dict` `closest[hướng] = (khoảng cách, quân)` và cập nhật khi gặp quân gần hơn. Phù hợp khi bàn cờ rất lớn (ví dụ `10^9 × 10^9`) mà số quân vừa phải — mô phỏng từng ô sẽ không chạy nổi.

In [10]:
def attack_group(queens, king):
    kx, ky = king
    closest = {}
    for x, y in queens:
        dx, dy = x - kx, y - ky
        if dx == 0 or dy == 0 or abs(dx) == abs(dy):
            d = (sign(dx), sign(dy))
            dist = max(abs(dx), abs(dy))
            if d not in closest or dist < closest[d][0]:
                closest[d] = (dist, [x, y])
    return [q for _, q in closest.values()]


for qs, k, want in examples:
    assert sorted(attack_group(qs, k)) == sorted(want)
print("Cách 4 đúng trên ví dụ")
print("ví dụ 2 ->", attack_group(*examples[1][:2]))

Cách 4 đúng trên ví dụ
ví dụ 2 -> [[2, 2], [3, 4], [4, 4]]


**Độ phức tạp:** Time `O(q)`, Space `O(1)` (tối đa 8 khoá) — tốt nhất về lý thuyết, và **không** phụ thuộc kích thước bàn cờ.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 (`king = (3,3)`):

| Hướng (dx,dy) | Các ô đi qua | Kết quả |
|---|---|---|
| (−1,−1) | (2,2) có hậu | ghi (2,2), dừng — (1,1), (0,0) bị chắn |
| (−1, 0) | (2,3), (1,3), (0,3), ra biên | không có |
| (−1, 1) | (2,4), (1,5), (0,6), ra biên | không có |
| (0,−1) | (3,2), (3,1), (3,0), ra biên | không có |
| (0, 1) | (3,4) có hậu | ghi (3,4) — (3,5) bị chắn |
| (1,−1) | (4,2), (5,1), (6,0), ra biên | không có |
| (1, 0) | (4,3), (5,3), (6,3), (7,3), ra biên | không có |
| (1, 1) | (4,4) có hậu | ghi (4,4) |

Hàm `trace_simulate` in đúng bảng này:

In [11]:
def trace_simulate(queens, king):
    qset = {tuple(q) for q in queens}
    kx, ky = king
    ans = []
    for dx, dy in DIRS8:
        x, y = kx + dx, ky + dy
        visited = []
        hit = None
        while 0 <= x < 8 and 0 <= y < 8:
            visited.append((x, y))
            if (x, y) in qset:
                hit = (x, y)
                break
            x += dx
            y += dy
        status = f"GẶP HẬU {hit}" if hit else "ra biên"
        print(f"hướng ({dx:>2},{dy:>2}): đi qua {visited} -> {status}")
        if hit:
            ans.append(list(hit))
    return ans


print(trace_simulate(*examples[1][:2]))

hướng (-1,-1): đi qua [(2, 2)] -> GẶP HẬU (2, 2)
hướng (-1, 0): đi qua [(2, 3), (1, 3), (0, 3)] -> ra biên
hướng (-1, 1): đi qua [(2, 4), (1, 5), (0, 6)] -> ra biên
hướng ( 0,-1): đi qua [(3, 2), (3, 1), (3, 0)] -> ra biên
hướng ( 0, 1): đi qua [(3, 4)] -> GẶP HẬU (3, 4)
hướng ( 1,-1): đi qua [(4, 2), (5, 1), (6, 0)] -> ra biên
hướng ( 1, 0): đi qua [(4, 3), (5, 3), (6, 3), (7, 3)] -> ra biên
hướng ( 1, 1): đi qua [(4, 4)] -> GẶP HẬU (4, 4)
[[2, 2], [3, 4], [4, 4]]


Trace Cách 4 — xem `closest` thay đổi khi lần lượt xét từng quân hậu:

In [12]:
def trace_group(queens, king):
    kx, ky = king
    closest = {}
    for x, y in queens:
        dx, dy = x - kx, y - ky
        if dx == 0 or dy == 0 or abs(dx) == abs(dy):
            d, dist = (sign(dx), sign(dy)), max(abs(dx), abs(dy))
            if d not in closest or dist < closest[d][0]:
                closest[d] = (dist, [x, y])
                note = "cập nhật"
            else:
                note = "bị chắn bởi quân gần hơn"
            print(f"hậu ({x},{y}): hướng {d}, khoảng cách {dist} -> {note}")
        else:
            print(f"hậu ({x},{y}): không cùng đường với vua")
    return [q for _, q in closest.values()]


print(trace_group(*examples[1][:2]))

hậu (0,0): hướng (-1, -1), khoảng cách 3 -> cập nhật
hậu (1,1): hướng (-1, -1), khoảng cách 2 -> cập nhật
hậu (2,2): hướng (-1, -1), khoảng cách 1 -> cập nhật
hậu (3,4): hướng (0, 1), khoảng cách 1 -> cập nhật
hậu (3,5): hướng (0, 1), khoảng cách 2 -> bị chắn bởi quân gần hơn
hậu (4,4): hướng (1, 1), khoảng cách 1 -> cập nhật
hậu (4,5): không cùng đường với vua
[[2, 2], [3, 4], [4, 4]]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def queensAttacktheKing(self, queens: List[List[int]], king: List[int]) -> List[List[int]]:
        qset = {(x, y) for x, y in queens}
        ans = []
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                if dx == 0 and dy == 0:
                    continue
                x, y = king[0] + dx, king[1] + dy
                while 0 <= x < 8 and 0 <= y < 8:
                    if (x, y) in qset:
                        ans.append([x, y])
                        break
                    x += dx
                    y += dy
        return ans


sol = Solution()
for qs, k, want in examples:
    got = sol.queensAttacktheKing(qs, k)
    print(got)
    assert sorted(got) == sorted(want)

[[0, 1], [1, 0], [3, 3]]
[[2, 2], [3, 4], [4, 4]]


## 10. Vì sao đúng

**Mệnh đề:** quân hậu `q` tấn công được vua ⇔ `q` là quân hậu **đầu tiên** gặp được khi đi từ vua theo hướng `(sign(dx), sign(dy))`.

- (⇐) Nếu `q` là quân đầu tiên trên một tia từ vua, các ô giữa đều trống (bất biến mục 6), và `q` cùng đường với vua → `q` tấn công được.
- (⇒) Nếu `q` tấn công được, `q` cùng hàng/cột/chéo với vua nên nằm trên đúng một trong 8 tia; không có quân nào giữa nên khi đi theo tia đó, ô có hậu đầu tiên chính là `q`.

Thuật toán thử đủ 8 tia, mỗi tia dừng ở quân đầu tiên (hoặc biên) → in ra đúng tập các quân tấn công được, mỗi quân một lần (mỗi tia chứa tối đa một quân được ghi). Vua không chắn gì vì tia bắt đầu từ ô kế bên vua.

Cách 4 đúng vì với mỗi tia, quân "đầu tiên" chính là quân có khoảng cách `max(|dx|, |dy|)` **nhỏ nhất** trên tia đó.

## 11. Time / Space Complexity

Gọi `q` = số quân hậu (`< 64`), `N = 8` là cạnh bàn cờ.

| Cách | Time | Space |
|---|---|---|
| 1. Kiểm tra từng cặp | `O(q²)` | `O(1)` |
| 2. Mô phỏng + set | `O(q + 8N)` | `O(q)` |
| 3. Mô phỏng + ma trận | `O(N² + q)` | `O(N²)` |
| 4. Gom theo hướng | `O(q)` | `O(1)` |

Với bàn cờ 8×8 mọi cách đều tức thì; khác biệt chỉ quan trọng khi tổng quát hoá lên bàn cờ lớn.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

Vì thứ tự tuỳ ý, ta so sánh sau khi **sắp xếp**.

In [14]:
methods = {
    "brute": attack_brute,
    "simulate": attack_simulate,
    "board": attack_board,
    "group": attack_group,
    "submit": lambda q, k: Solution().queensAttacktheKing(q, k),
}

all_but_king = [[x, y] for x in range(8) for y in range(8) if [x, y] != [4, 4]]
edge = [
    ([[7, 7]], [0, 0], [[7, 7]]),                          # chéo dài nhất
    ([[0, 7]], [7, 0], [[0, 7]]),                          # chéo phụ dài nhất
    ([[1, 2]], [0, 0], []),                                # nước mã — không tấn công
    ([[0, 1], [0, 2], [0, 3]], [0, 0], [[0, 1]]),          # chắn nhau trên một hàng
    (all_but_king, [4, 4], [[3, 3], [3, 4], [3, 5], [4, 3], [4, 5], [5, 3], [5, 4], [5, 5]]),  # tối đa 8
]
for name, f in methods.items():
    for qs, k, want in examples + edge:
        assert sorted(f(qs, k)) == sorted(want), (name, qs, k)
print("Mọi cách đúng trên ví dụ + edge case")

Mọi cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Ngoài so với brute force, ta còn kiểm tra bằng một **hàm kiểm tra hợp lệ** độc lập: một quân tấn công được khi cùng đường và mọi ô ở giữa đều trống.

In [15]:
def is_attacker(q, king, qset):
    dx, dy = q[0] - king[0], q[1] - king[1]
    if not (dx == 0 or dy == 0 or abs(dx) == abs(dy)):
        return False
    sx, sy, dist = sign(dx), sign(dy), max(abs(dx), abs(dy))
    return all((king[0] + t * sx, king[1] + t * sy) not in qset for t in range(1, dist))


rng = random.Random(1222)
cells = [(x, y) for x in range(8) for y in range(8)]
for _ in range(500):
    pts = rng.sample(cells, rng.randint(2, 64))
    king = list(pts[0])
    queens = [list(p) for p in pts[1:]]
    qset = {tuple(q) for q in queens}
    truth = sorted(q for q in queens if is_attacker(q, king, qset))
    for name, f in methods.items():
        assert sorted(f(queens, king)) == truth, name
print("500 bàn cờ ngẫu nhiên: mọi cách khớp hàm kiểm tra")

500 bàn cờ ngẫu nhiên: mọi cách khớp hàm kiểm tra


### 12.3 Test giới hạn

Bàn cờ có 63 quân hậu (đầy trừ ô vua) với vua ở mọi vị trí; đo thời gian.

In [16]:
t0 = time.perf_counter()
for kx in range(8):
    for ky in range(8):
        qs = [[x, y] for x in range(8) for y in range(8) if (x, y) != (kx, ky)]
        res = Solution().queensAttacktheKing(qs, [kx, ky])
        # số hướng còn nằm trong bàn cờ = số quân tấn công
        expect = sum(0 <= kx + dx < 8 and 0 <= ky + dy < 8 for dx, dy in DIRS8)
        assert len(res) == expect
print(f"64 vị trí vua trên bàn cờ đầy: OK trong {(time.perf_counter() - t0) * 1000:.1f} ms")

64 vị trí vua trên bàn cờ đầy: OK trong 1.5 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên `break`: lấy cả các quân bị chắn

In [17]:
def wrong_no_break(queens, king):
    qset = {tuple(q) for q in queens}
    ans = []
    for dx, dy in DIRS8:
        x, y = king[0] + dx, king[1] + dy
        while 0 <= x < 8 and 0 <= y < 8:
            if (x, y) in qset:
                ans.append([x, y])      # không dừng!
            x += dx
            y += dy
    return ans


qs, k, want = examples[0]
print("sai :", sorted(wrong_no_break(qs, k)))
print("đúng:", sorted(want))

sai : [[0, 1], [0, 4], [1, 0], [3, 3], [4, 0]]
đúng: [[0, 1], [1, 0], [3, 3]]


**Vì sao sai:** `(0,4)` và `(4,0)` bị `(0,1)`, `(1,0)` chắn nhưng vẫn được ghi. **Sửa:** `break` ngay khi gặp quân đầu tiên trên mỗi hướng.

### Lỗi 2 — Chỉ xét 4 hướng (quên đường chéo) hoặc chỉ xét chéo chính

In [18]:
def wrong_only_main_diag(queens, king):
    kx, ky = king
    closest = {}
    for x, y in queens:
        dx, dy = x - kx, y - ky
        if dx == 0 or dy == 0 or dx == dy:             # quên dx == -dy (chéo phụ)
            d, dist = (sign(dx), sign(dy)), max(abs(dx), abs(dy))
            if d not in closest or dist < closest[d][0]:
                closest[d] = (dist, [x, y])
    return [q for _, q in closest.values()]


qs, k = [[0, 7], [5, 5]], [7, 0]
print("sai :", sorted(wrong_only_main_diag(qs, k)))
print("đúng:", sorted(attack_simulate(qs, k)))

sai : []
đúng: [[0, 7]]


**Vì sao sai:** hậu ở `(0,7)` nằm trên **chéo phụ** của vua `(7,0)` (`dx = −7`, `dy = 7`). Điều kiện `dx == dy` chỉ bắt chéo chính. **Sửa:** `abs(dx) == abs(dy)`, hoặc dùng đủ 8 vector hướng.

### Lỗi 3 — Chuẩn hoá hướng bằng `abs` hoặc bằng chia không dấu

In [19]:
def wrong_abs_direction(queens, king):
    kx, ky = king
    closest = {}
    for x, y in queens:
        dx, dy = x - kx, y - ky
        if dx == 0 or dy == 0 or abs(dx) == abs(dy):
            d = (abs(sign(dx)), abs(sign(dy)))          # mất thông tin trái/phải, trên/dưới
            dist = max(abs(dx), abs(dy))
            if d not in closest or dist < closest[d][0]:
                closest[d] = (dist, [x, y])
    return [q for _, q in closest.values()]


qs, k, want = examples[1]
print("sai :", sorted(wrong_abs_direction(qs, k)))
print("đúng:", sorted(want))

sai : [[2, 2], [3, 4]]
đúng: [[2, 2], [3, 4], [4, 4]]


**Vì sao sai:** `(2,2)` (hướng `(−1,−1)`) và `(4,4)` (hướng `(1,1)`) bị gộp làm một nhóm `(1,1)` → chỉ giữ được một quân. Hai phía của vua là **hai tia khác nhau**. **Sửa:** giữ dấu: `(sign(dx), sign(dy))`.

### Lỗi 4 — Kiểm tra ra biên sai khi dùng vòng `while` kiểu Cách 3

Nếu viết `while not board[x][y]` mà kiểm tra biên **sau**, Python với chỉ số âm sẽ lấy ô ở đầu kia bàn cờ (không báo lỗi), còn chỉ số `8` thì `IndexError`.

In [20]:
board = [[False] * 8 for _ in range(8)]
board[0][7] = True
print("board[0][-1] =", board[0][-1], "<- chỉ số -1 'vòng' sang cột 7, không báo lỗi!")
try:
    board[8][0]
except IndexError as e:
    print("board[8][0]:", e)

board[0][-1] = True <- chỉ số -1 'vòng' sang cột 7, không báo lỗi!
board[8][0]: list index out of range


**Sửa:** luôn kiểm tra `0 <= x < 8 and 0 <= y < 8` **trước** khi đọc `board[x][y]` (nhờ tính "ngắn mạch" của `and`).

## 14. Mẹo & nhận dạng pattern

- Quân cờ "trượt" (hậu, xe, tượng) → **8 (hoặc 4) vector hướng** + vòng `while` bước từng ô + `break` khi gặp vật cản.
- "Chỉ quân gần nhất mới có tác dụng" → hoặc **mô phỏng từ tâm ra**, hoặc **gom nhóm theo hướng giữ min khoảng cách**.
- Chuẩn hoá hướng bằng `sign`, khoảng cách theo **Chebyshev** `max(|dx|, |dy|)`.
- Thứ tự kết quả tuỳ ý → khi test luôn `sorted`.
- Toạ độ làm khoá → dùng **tuple**.

## 15. Biến thể

1. **Xe (rook)** chỉ 4 hướng thẳng; **tượng (bishop)** chỉ 4 hướng chéo — đổi danh sách hướng.
2. **Bàn cờ `n × n` khổng lồ** (`n` tới `10^9`) → dùng Cách 4 (không mô phỏng từng ô).
3. **Available Captures for Rook (LeetCode 999):** xe đi 4 hướng, bị chắn bởi tượng của mình, đếm số tốt bắt được.
4. **Vua có bị chiếu không?** — kiểm tra có ít nhất một quân tấn công được.
5. **Các ô vua có thể đi tới an toàn**: với mỗi ô trong 8 ô xung quanh, kiểm tra có bị hậu nào tấn công không (cẩn thận: vua di chuyển có thể "mở" đường cho hậu).

In [21]:
def attackers_generic(pieces, target, dirs, size):
    """Biến thể 1-2: quân trượt theo 'dirs' trên bàn cờ size x size, dùng gom nhóm."""
    tx, ty = target
    dirset = set(dirs)
    closest = {}
    for x, y in pieces:
        dx, dy = x - tx, y - ty
        if dx == 0 or dy == 0 or abs(dx) == abs(dy):
            d, dist = (sign(dx), sign(dy)), max(abs(dx), abs(dy))
            if d in dirset and (d not in closest or dist < closest[d][0]):
                closest[d] = (dist, [x, y])
    return sorted(q for _, q in closest.values())


ROOK = [(1, 0), (-1, 0), (0, 1), (0, -1)]
BISHOP = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
pieces = [[0, 0], [3, 7], [10 ** 9 - 1, 3], [6, 6]]
print("xe   :", attackers_generic(pieces, [3, 3], ROOK, 10 ** 9))
print("tượng:", attackers_generic(pieces, [3, 3], BISHOP, 10 ** 9))

xe   : [[3, 7], [999999999, 3]]
tượng: [[0, 0], [6, 6]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra từng cặp | mỗi hậu kiểm tra vật cản bằng mọi hậu khác | `O(q²)` | `O(1)` | trọng tài |
| 2. Mô phỏng + set | từ vua bắn 8 tia, dừng ở hậu đầu tiên | `O(q + 8N)` | `O(q)` | **nộp bài**, dễ hiểu |
| 3. Mô phỏng + ma trận | như 2, tra bằng `board` | `O(N² + q)` | `O(N²)` | bàn cờ nhỏ cố định |
| 4. Gom theo hướng | giữ hậu gần nhất mỗi hướng | `O(q)` | `O(1)` | bàn cờ khổng lồ |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`queens = [[5,6],[7,7],[2,1],[0,7],[1,6],[5,1],[3,7],[0,3],[4,0],[1,2],[6,3],[5,0],[0,4],[2,2],[1,1],[6,4],[5,4],[0,0],[2,6],[4,5],[5,2],[1,4],[7,5],[2,3],[0,5],[4,2],[1,0],[2,7],[0,1],[4,6],[6,1],[0,6],[4,3],[1,7]]`, `king = [3,4]` — đáp án có bao nhiêu quân? (Gợi ý: đây là ví dụ 3 cũ của LeetCode.)

### Bài 2 (tính tay)

Vua `(4,4)`, hậu ở `(4,0), (4,1), (1,1), (7,7), (6,2), (0,4)`. Quân nào tấn công được?

### Bài 3 (điền TODO) — vua có bị chiếu không?

In [22]:
def is_in_check(queens, king):
    """True nếu có ít nhất một hậu tấn công được vua. Dừng ngay khi tìm thấy."""
    qset = {tuple(q) for q in queens}
    # TODO: duyệt 8 hướng, bước từng ô; gặp hậu -> return True
    raise NotImplementedError

# print(is_in_check([[1, 2]], [0, 0]), is_in_check([[5, 5]], [0, 0]))   # mong đợi False True

### Bài 4 (tự viết lại)

Viết lại Cách 4 (gom theo hướng) **không dùng** hàm `sign`, mà dùng phép chia nguyên cho khoảng cách: `d = (dx // dist, dy // dist)`. Tại sao phép chia này đúng?

In [23]:
def attack_group_div_todo(queens, king):
    # TODO
    raise NotImplementedError

# print(sorted(attack_group_div_todo([[0,1],[1,0],[4,0],[0,4],[3,3],[2,4]], [0, 0])))   # [[0,1],[1,0],[3,3]]

### Bài 5 (mini challenge) — ô an toàn cho vua

Trả về danh sách các ô (trong 8 ô xung quanh vua, nằm trong bàn cờ, **không có hậu**) mà nếu vua bước tới thì **không** bị hậu nào tấn công. Lưu ý: khi vua rời ô cũ, ô cũ trở thành trống.

In [24]:
def safe_moves_todo(queens, king):
    # TODO
    raise NotImplementedError

# print(safe_moves_todo([[0, 2]], [0, 0]))   # mong đợi [[1, 0]]  ((0,1),(1,1) bị hậu (0,2) tấn công)

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: chạy `trace_simulate`. Mức 2: vua ở `(3,4)` có đủ 8 hướng trong bàn cờ. Mức 3: đếm số hướng gặp hậu.

**Bài 2.** Mức 1: `(4,0)` bị `(4,1)` chắn. Mức 2: `(1,1)` trên chéo `(−1,−1)` — ô giữa `(3,3), (2,2)` trống. Mức 3: `(6,2)` có `dx = 2, dy = −2` → chéo phụ, ô giữa `(5,3)` trống. `(5,5)`, `(6,6)` trống nên `(7,7)` tấn công được. `(0,4)` cùng cột, ô giữa trống.

**Bài 3.** Mức 1: giống Cách 2. Mức 2: thay `ans.append` bằng `return True`. Mức 3: hết 8 hướng thì `return False`.

**Bài 4.** Mức 1: `dist = max(abs(dx), abs(dy))`. Mức 2: trên hàng/cột/chéo, `|dx|` và `|dy|` hoặc bằng `0` hoặc bằng `dist`. Mức 3: nên `dx // dist ∈ {−1, 0, 1}` chính xác (chia hết).

**Bài 5.** Mức 1: thử 8 ô quanh vua. Mức 2: bỏ ô ngoài biên và ô có hậu (vua ăn hậu là một luật khác — ở đây bỏ qua). Mức 3: với ô mới, gọi `is_in_check(queens, new_king)` — vua cũ không còn ở đó nên không cần làm gì thêm.

## 19. Đáp án tham khảo

In [25]:
# Bài 1, 2
q1 = [[5,6],[7,7],[2,1],[0,7],[1,6],[5,1],[3,7],[0,3],[4,0],[1,2],[6,3],[5,0],[0,4],[2,2],[1,1],[6,4],[5,4],[0,0],[2,6],[4,5],[5,2],[1,4],[7,5],[2,3],[0,5],[4,2],[1,0],[2,7],[0,1],[4,6],[6,1],[0,6],[4,3],[1,7]]
r1 = sorted(Solution().queensAttacktheKing(q1, [3, 4]))
print(len(r1), r1)
assert r1 == sorted(attack_brute(q1, [3, 4]))
r2 = sorted(Solution().queensAttacktheKing([[4, 0], [4, 1], [1, 1], [7, 7], [6, 2], [0, 4]], [4, 4]))
print(r2)
assert r2 == [[0, 4], [1, 1], [4, 1], [6, 2], [7, 7]]

7 [[1, 4], [1, 6], [2, 3], [3, 7], [4, 3], [4, 5], [5, 4]]
[[0, 4], [1, 1], [4, 1], [6, 2], [7, 7]]


In [26]:
def is_in_check(queens, king):
    qset = {tuple(q) for q in queens}
    for dx, dy in DIRS8:
        x, y = king[0] + dx, king[1] + dy
        while 0 <= x < 8 and 0 <= y < 8:
            if (x, y) in qset:
                return True
            x += dx
            y += dy
    return False


def attack_group_div(queens, king):
    kx, ky = king
    closest = {}
    for x, y in queens:
        dx, dy = x - kx, y - ky
        if dx == 0 or dy == 0 or abs(dx) == abs(dy):
            dist = max(abs(dx), abs(dy))
            d = (dx // dist, dy // dist)
            if d not in closest or dist < closest[d][0]:
                closest[d] = (dist, [x, y])
    return [q for _, q in closest.values()]


def safe_moves(queens, king):
    qset = {tuple(q) for q in queens}
    out = []
    for dx, dy in DIRS8:
        x, y = king[0] + dx, king[1] + dy
        if 0 <= x < 8 and 0 <= y < 8 and (x, y) not in qset and not is_in_check(queens, [x, y]):
            out.append([x, y])
    return sorted(out)


assert (is_in_check([[1, 2]], [0, 0]), is_in_check([[5, 5]], [0, 0])) == (False, True)
assert sorted(attack_group_div(*examples[0][:2])) == sorted(examples[0][2])
assert safe_moves([[0, 2]], [0, 0]) == [[1, 0]]
rng = random.Random(3)
for _ in range(300):
    pts = rng.sample(cells, rng.randint(2, 20))
    king, queens = list(pts[0]), [list(p) for p in pts[1:]]
    assert is_in_check(queens, king) == bool(attack_brute(queens, king))
    assert sorted(attack_group_div(queens, king)) == sorted(attack_brute(queens, king))
    for mv in safe_moves(queens, king):
        assert not any(is_attacker(q, mv, {tuple(p) for p in queens}) for q in queens)
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
qset = {(x, y) for x, y in queens}
ans = []
for dx in (-1, 0, 1):
    for dy in (-1, 0, 1):
        if dx == dy == 0: continue
        x, y = kx + dx, ky + dy
        while 0 <= x < 8 and 0 <= y < 8:      # kiểm tra biên TRƯỚC
            if (x, y) in qset:
                ans.append([x, y]); break     # quân đầu tiên chắn phần còn lại
            x += dx; y += dy
return ans
```

**Checklist:**
- [ ] Đủ **8** hướng (cả chéo chính và chéo phụ)?
- [ ] `break` sau quân đầu tiên trên mỗi hướng?
- [ ] Kiểm tra biên trước khi đọc ô?
- [ ] Hướng giữ **dấu** (không dùng `abs`)?
- [ ] Đổi list → tuple khi cho vào `set`, và trả về list `[x, y]`?
- [ ] Khi tự test: so sánh sau khi `sorted` (thứ tự tuỳ ý)?